<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/workflow-guides/workflow-02-analyze-apparent-fishing-effort-argentinian-eez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyze apparent fishing effort in Argentinian EEZ

This guide provides detailed instructions to on how to use the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client) to **Analyze apparent fishing effort in [Argentinian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8466) region and monitor industrial trawlers** using **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)**, **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)**, and **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)**.

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Data Caveats](https://globalfishingwatch.org/our-apis/documentation#data-caveat), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be easily installed using pip:

In [1]:
# %pip install gfw-api-python-client

## Usage

In [2]:
import os
import datetime
import pandas as pd
import gfwapiclient as gfw

In [3]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception as exc:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [4]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Introduction

**Use Case: A Fisheries Enforcement Officer Monitoring Industrial Trawlers**

Maria, a fisheries enforcement officer in Argentina, monitors industrial trawlers operating within **[Argentinian Exclusive Economic Zone (EEZ)](https://www.marineregions.org/gazetteer.php?p=details&id=8466)**. His goal is to:

1. Analyzing apparent fishing effort for **trawlers** operating in Argentinian EEZ.
2. Identifying vessels involved in **apparent trawling activity** and determining their reported **flag states**.
3. Checking vessel history, including **potential transshipment** and **port visits**.
4. Generating reports to support fisheries enforcement decisions.

**APIs Used:**
️
1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** – Retrieve **apparent fishing effort** data for trawlers.
2. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** – Group vessels by ID that are involved in **trawling activity**.
3. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** – Retrieve **vessel identity** & **ownership** details.
4. **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)** – Fetch **port visits** & **potential transshipment** history. 

**Important:** In order to avoid any misinterpretation of **GFW data**, please refer to our official **data caveats** documentations:
- [Apparent fishing effort](https://globalfishingwatch.org/dataset-and-code-fishing-effort/) 
- [Exclusive economic zone boundaries definition](https://globalfishingwatch.org/our-apis/documentation#exclusive-economic-zone-boundaries-definition)
- [Vessel ID](https://globalfishingwatch.org/our-apis/documentation#vessel-id)
- [Vessel API - Vessel identity information](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information)

**Important Caveats:**

1. The [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api) only supports **one active report per user at a time**.
2. **Sending multiple requests simultaneously** results in a **429 Too Many Requests** error.
3. If a report takes over **100 seconds** to generate, it may return a **524 Gateway Timeout** error.

## Step 0: Identify the Region of Interest (ROI) - Argentinian EEZ

Before making API requests, Maria must specify the geographic area for analysis using a **Region ID**:

**Options to Define the Region:**

1. **Using Region ID** - Each EEZ has a unique ID in the **[public-eez-areas](https://globalfishingwatch.org/our-apis/documentation#regions)** dataset.
2. **Custom Geometries** - Users can define a custom area using GeoJSON.
   
For **[Argentinian EEZ, the region ID is 8466](https://www.marineregions.org/gazetteer.php?p=details&id=8466)** (public-eez-areas dataset).

## Step 1: Retrieve Apparent Fishing Effort in Argentinian EEZ

Maria **first queries** the **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** to get **apparent fishing effort for all vessels**, grouping them by **gear type** in **[Argentinian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8466)**. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - 8466 Argentinian EEZ
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 6 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Gear Type

**Why This Step?**

- Identifies which gear types (e.g., `trawlers`, `squid jiggers` etc.) are most active in the Argentinian EEZ.
- Establishes baseline fishing activity trends before narrowing the search to specific vessels.

In [5]:
step_1_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="HIGH",
    group_by="GEARTYPE",
    temporal_resolution="MONTHLY",
    start_date="2024-08-01",
    end_date="2025-01-31",
    spatial_aggregation=True,
    region={
        "dataset": "public-eez-areas",
        "id": "8466",
    },
)

In [6]:
step_1_report_df = step_1_report_result.df()

In [7]:
step_1_report_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     40 non-null     object 
 1   detections               0 non-null      object 
 2   flag                     0 non-null      object 
 3   gear_type                40 non-null     object 
 4   hours                    40 non-null     float64
 5   vessel_ids               40 non-null     int64  
 6   vessel_id                0 non-null      object 
 7   vessel_type              0 non-null      object 
 8   entry_timestamp          0 non-null      object 
 9   exit_timestamp           0 non-null      object 
 10  first_transmission_date  0 non-null      object 
 11  last_transmission_date   0 non-null      object 
 12  imo                      0 non-null      object 
 13  mmsi                     0 non-null      object 
 14  call_sign                0 n

In [8]:
step_1_report_df[["gear_type", "hours", "vessel_ids"]].head()

,gear_type,hours,vessel_ids
0,inconclusive,215.623889,2
1,set_longlines,17.925833,1
2,inconclusive,98.155833,2
3,other_purse_seines,57.377500,1
4,set_longlines,153.960278,3


In [9]:
step_1_agg_report_df = (
    step_1_report_df.groupby(["gear_type"], as_index=False)
    .agg(hours=("hours", "sum"), vessel_ids=("vessel_ids", "sum"))
    .sort_values(by="hours", ascending=False)
)

In [10]:
step_1_agg_report_df.head()

,gear_type,hours,vessel_ids
8,trawlers,240642.414444,1315
1,fishing,16023.201389,94
7,squid_jigger,6124.062222,44
2,fixed_gear,1948.325556,16
3,inconclusive,1072.351389,14


### What We have Learned from Step 1

- Multiple **gear types** were potentially detected in Argentinian EEZ.
- `Trawlers` appear to be operating, but further **vessel-level investigation** is needed.

## Step 2: Retrieve Vessel IDs for Trawlers

Maria refines her **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** request to group by **vessel ID**, and filtering only for **trawlers** in **[Argentinian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8466)**. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - [8466 Argentinian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8466)
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 6 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Vessel ID
4. **[Gear Type](https://globalfishingwatch.org/our-apis/documentation#gear-types-supported)** - Trawlers 

**Why Use group-by=VESSEL_ID?**

Grouping by **VESSEL_ID** allows **individual vessel identification** in the response. This is crucial for **tracking vessel activity** and, more importantly, linking each detected vessel to the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** in the next step. By structuring the query this way, we can fetch vessel details such as **flag, name, and ownership records** in **Step 3 below**.


In [11]:
step_2_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="HIGH",
    group_by="VESSEL_ID",
    temporal_resolution="ENTIRE",
    filters=["geartype in ('trawlers')"],
    start_date="2024-08-01",
    end_date="2025-01-31",
    spatial_aggregation=True,
    region={
        "dataset": "public-eez-areas",
        "id": "8466",
    },
)

In [12]:
step_2_report_df = step_2_report_result.df()

In [13]:
step_2_report_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 385 entries, 0 to 384
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   date                     385 non-null    object             
 1   detections               0 non-null      object             
 2   flag                     385 non-null    object             
 3   gear_type                385 non-null    object             
 4   hours                    385 non-null    float64            
 5   vessel_ids               0 non-null      object             
 6   vessel_id                385 non-null    object             
 7   vessel_type              385 non-null    object             
 8   entry_timestamp          385 non-null    datetime64[ns, UTC]
 9   exit_timestamp           385 non-null    datetime64[ns, UTC]
 10  first_transmission_date  385 non-null    datetime64[ns, UTC]
 11  last_transmission_date   385 non

In [14]:
step_2_report_df[["flag", "gear_type", "hours", "mmsi", "ship_name"]].head()

,flag,gear_type,hours,mmsi,ship_name
0,URY,TRAWLERS,10.023056,770576463,KALATXORI
1,ARG,TRAWLERS,326.160833,701079000,ENTRENA UNO
2,ARG,TRAWLERS,714.842500,701000882,FELIX AUGUSTO
3,ARG,TRAWLERS,641.522222,701000932,ANTONIO ALVAREZ
4,ARG,TRAWLERS,295.007500,701000820,CORAJE


### Explore Vessels Potentially Engaged in Trawling Activity in the Argentinian EEZ

In [15]:
step_2_agg_report_df = (
    step_2_report_df.groupby(["flag", "gear_type", "mmsi", "ship_name"], as_index=False)
    .agg(hours=("hours", "sum"))
    .sort_values(by="hours", ascending=False)
)

In [16]:
step_2_agg_report_df.head()

,flag,gear_type,mmsi,ship_name,hours
297,ARG,TRAWLERS,701024000,ATLANTIC SURF III,3151.855278
247,ARG,TRAWLERS,701006605,CAPESANTE,2270.097500
22,ARG,TRAWLERS,701000577,MISS TIDE,2244.895833
296,ARG,TRAWLERS,701023000,CAROLINA P,2065.586944
301,ARG,TRAWLERS,701037000,DON PEDRO,1807.146111


### What We have Learned from Step 2

- There are vessels appear to have been engaged in potential trawling activity in Argentinian EEZ over the past 6 months i.e.,:
  - `ATLANTIC SURF III (mmsi: 701024000, flag: ARG)`
  - `CAPESANTE (mmsi: 701006605, flag: ARG)`
- We will retrieve these vessels' `ownership`, `flag history`, and `authorizations` in **Step 3 to validate** them.

## Step 3: Retrieve Vessel Details Using the Vessels API

Maria queries the **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** to **get detailed vessel identity and ownership records**. Please [learn more about Vessels API here](https://globalfishingwatch.org/our-apis/documentation#vessels-api) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#vessel-api-vessel-identity-information).

**Filters Used:**

1. **Vessel IDs** from [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api), **Step 2 above**.
2. **[Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset)** - `public-global-vessel-identity:latest`.
3. **[Includes](https://globalfishingwatch.org/our-apis/documentation#get-vessels-by-ids-url-parameters)** - `POTENTIAL_RELATED_SELF_REPORTED_INFO`.

**Note:** Vessels may change identifiers over time, such as their `Maritime Mobile Service Identity (MMSI)`,` International Maritime Organization (IMO) number)`, `call sign`, or even their `name`. These changes can occur due to `re-registration`, `changes in ownership`, or other `operational reasons` within the `AIS transponder`. Parameter (`includes = POTENTIAL_RELATED_SELF_REPORTED_INFO`) helps group all **vessel ids** that are **potentially related** as part of the **same physical vessel** based on publicly available registry information.

In [17]:
step_2_vessel_mmsis = list(step_2_agg_report_df["mmsi"].head(n=2))

In [18]:
step_2_vessel_mmsis

['701024000', '701006605']

In [19]:
step_2_vessel_ids = list(
    step_2_report_df[step_2_report_df["mmsi"].isin(step_2_vessel_mmsis)][
        "vessel_id"
    ].unique()
)

In [20]:
step_2_vessel_ids

['de8a03acd-dc6c-8e08-2867-24e55ffc0017',
 '8e930bac5-594b-aa3f-081d-d12668819e1f']

In [21]:
step_3_vessels_result = await gfw_client.vessels.get_vessels_by_ids(
    ids=step_2_vessel_ids,
)

In [22]:
step_3_vessels_df = step_3_vessels_result.df()

In [23]:
step_3_vessels_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   dataset                         2 non-null      object
 1   registry_info_total_records     2 non-null      int64 
 2   registry_info                   2 non-null      object
 3   registry_owners                 2 non-null      object
 4   registry_public_authorizations  2 non-null      object
 5   combined_sources_info           2 non-null      object
 6   self_reported_info              2 non-null      object
dtypes: int64(1), object(6)
memory usage: 244.0+ bytes


**Understanding Vessel Details Response Data**

- **registryInfoTotalRecords** – This represents the **number of registry records** found for the vessels.
- **registryInfo** – Contains **public registry data**. This data is sourced from official **vessel registries**.
- **registryOwners** – Lists the **registered owners** of the vessel based on public sources.
- **registryPublicAuthorizations** – Represents known **fishing authorizations** from public sources. Users should verify against national registries and RFMO records for additional context.
- **combinedSourcesInfo** – Provides inferred data from multiple sources, including. This is not explicitly reported by vessels but determined through **GFW's classification methods**.
- **selfReportedInfo** – Contains **AIS self-reported** data, including `MMSI`, `ship name`, and `flag` as broadcast by the **vessel itself**. Self-reported data may not always align with registry data and should be cross-checked.

In [24]:
step_3_vessels_df[["registry_info", "registry_owners", "self_reported_info"]]

,registry_info,registry_owners,self_reported_info
0,"[{'id': '45502524c9a150e77869ee647423dba1', 's...","[{'name': 'GLACIAR PESQUERA', 'flag': 'ARG', '...",[{'id': '8e930bac5-594b-aa3f-081d-d12668819e1f...
1,"[{'id': '2d939efefd3f45788ed103ff0723f564', 's...","[{'name': 'CLEARWATER SEAFOODS', 'flag': 'CAN'...",[{'id': 'de8a03acd-dc6c-8e08-2867-24e55ffc0017...


### Explore Vessels Registry Info

In [25]:
step_3_registry_info_df = pd.json_normalize(
    step_3_vessels_df["registry_info"].explode()
)

In [26]:
step_3_registry_info_df[
    ["ssvid", "flag", "ship_name", "n_ship_name", "gear_types", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,gear_types,source_code
0,701024000,ARG,ATLANTIC SURF III,ATLANTICSURF3,[TRAWLERS],"[IMO, TMT_OTHER_OFFICIAL]"
1,701006605,ARG,CAPESANTE,CAPESANTE,[TRAWLERS],"[GFW-REVIEW, IMO, RESEARCH-PAPER, TMT_OTHER_OF..."
2,316003980,CAN,ATLANTICLEADER,ATLANTICLEADER,[TRAWLERS],"[IMO, TMT_OTHER_OFFICIAL]"


### Explore Registry Owners

In [27]:
step_3_registry_owners_df = pd.json_normalize(
    step_3_vessels_df["registry_owners"].explode()
)

In [28]:
step_3_registry_owners_df[["ssvid", "flag", "name", "source_code"]]

,ssvid,flag,name,source_code
0,701024000,ARG,GLACIAR PESQUERA,[TMT_OTHER_OFFICIAL]
1,701006605,CAN,CLEARWATER SEAFOODS,[RESEARCH-PAPER]
2,316003980,CAN,CS MANPAR,[TMT_OTHER_OFFICIAL]


### Explore Vessels Self Reported Info

In [29]:
step_3_self_reported_info_df = pd.json_normalize(
    step_3_vessels_df["self_reported_info"].explode()
)

In [30]:
step_3_self_reported_info_df[
    ["ssvid", "flag", "ship_name", "n_ship_name", "source_code"]
]

,ssvid,flag,ship_name,n_ship_name,source_code
0,701024000,ARG,ATLANTIC SURF III,ATLANTICSURF3,[AIS]
1,701006605,ARG,CAPESANTE,CAPESANTE,[AIS]
2,316003980,CAN,ATLANTIC LEADER,ATLANTICLEADER,[AIS]


### What We have Learned from Step 3

- **Vessel Identity:**
  - `ATLANTIC SURF III (mmsi: 701024000, flag: ARG)`- appears to be registered under Argentina (ARG)
  - `CAPESANTE (mmsi: 701006605, flag: ARG)` - appears to be registered under Argentina (ARG)
- **Ownership & Historical Changes:**
  - `ATLANTIC SURF III (mmsi: 701024000, flag: ARG)` - **GLACIAR PESQUERA** appears to be listed as the registered owner.
  - `CAPESANTE (mmsi: 701006605, flag: ARG)`- **CLEARWATER SEAFOODS** appears to be listed as the registered owner.

## Step 4: Detect Potential Port Visits, Encounters, or Fishing Events

Now Maria checks `port visits`, `encounters`, and `fishing events` using the **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)**, which allows **monitoring of vessel activities** such as `potential transshipments`, `unauthorized port entries`, or `fishing activity patterns`.

**Filters Used:**

1. **Vessel ID** from 4Wings API
2. **[Event Types](https://globalfishingwatch.org/our-apis/documentation#events-post-body-parameters)** - Port visits, encounters (potential transshipment), and fishing events.
3. **Time Range** - Last 6 months.
4. **[Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset)**:
   - `public-global-port-visits-events::latest` (Port Visits)
   - `public-global-encounters-events:latest` (Encounters between vessels)
   - `public-global-fishing-events:latest` (Fishing activity)
5. **[Encounter Types](https://globalfishingwatch.org/our-apis/documentation#events-post-body-parameters)** - CARRIER-FISHING

In [31]:
step_4_events_result = await gfw_client.events.get_all_events(
    datasets=[
        "public-global-encounters-events:latest",
        "public-global-fishing-events:latest",
        "public-global-port-visits-events:latest",
    ],
    vessels=step_2_vessel_ids,
    types=["ENCOUNTER", "FISHING", "PORT_VISIT"],
    start_date="2024-08-01",
    end_date="2025-01-31",
    encounter_types=["CARRIER-FISHING"],
    sort="-start",
)

In [32]:
step_4_events_df = step_4_events_result.df()

In [33]:
step_4_events_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 359 entries, 0 to 358
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   start         359 non-null    datetime64[ns, UTC]
 1   end           359 non-null    datetime64[ns, UTC]
 2   id            359 non-null    object             
 3   type          359 non-null    object             
 4   position      359 non-null    object             
 5   regions       359 non-null    object             
 6   bounding_box  359 non-null    object             
 7   distances     359 non-null    object             
 8   vessel        359 non-null    object             
 9   encounter     0 non-null      object             
 10  fishing       351 non-null    object             
 11  gap           0 non-null      object             
 12  loitering     0 non-null      object             
 13  port_visit    8 non-null      object             
dtypes: datetim

In [34]:
step_4_events_df["type"].value_counts()

type
fishing       351
port_visit      8
Name: count, dtype: int64

### Explore Apparent Fishing Events

In [35]:
step_4_fishing_events_df = step_4_events_df[step_4_events_df["fishing"].notna()]

In [36]:
step_4_fishing_df = pd.concat(
    [
        pd.json_normalize(step_4_fishing_events_df["vessel"], sep="_"),
        pd.json_normalize(step_4_fishing_events_df["fishing"], sep="_"),
    ],
    axis=1,
)

In [37]:
step_4_fishing_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351 entries, 0 to 350
Data columns (total 12 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   id                                  351 non-null    object 
 1   name                                351 non-null    object 
 2   ssvid                               351 non-null    object 
 3   flag                                351 non-null    object 
 4   type                                351 non-null    object 
 5   public_authorizations               351 non-null    object 
 6   nextPort                            0 non-null      object 
 7   total_distance_km                   351 non-null    float64
 8   average_speed_knots                 351 non-null    float64
 9   average_duration_hours              0 non-null      object 
 10  potential_risk                      351 non-null    bool   
 11  vessel_public_authorization_status  351 non-n

In [38]:
step_4_fishing_df[
    [
        "name",
        "ssvid",
        "total_distance_km",
        "average_speed_knots",
    ]
]

,name,ssvid,total_distance_km,average_speed_knots
0,ATLANTIC SURF III,701024000,117.545450,4.351747
1,ATLANTIC SURF III,701024000,10.408851,3.791667
2,ATLANTIC SURF III,701024000,61.366911,4.448980
3,ATLANTIC SURF III,701024000,92.180704,4.485407
4,ATLANTIC SURF III,701024000,974.761871,4.051429
...,...,...,...,...
346,CAPESANTE,701006605,95.128059,4.202542
347,CAPESANTE,701006605,205.826282,4.136848
348,CAPESANTE,701006605,369.336603,3.938255
349,ATLANTIC SURF III,701024000,256.908842,4.186333


In [39]:
step_4_fishing_df["ssvid"].value_counts()

ssvid
701024000    245
701006605    106
Name: count, dtype: int64

### Explore Port Visit Events

In [40]:
step_4_port_visit_events_df = step_4_events_df[step_4_events_df["port_visit"].notna()]

In [41]:
step_4_port_visits_df = pd.concat(
    [
        pd.json_normalize(step_4_port_visit_events_df["vessel"], sep="_"),
        pd.json_normalize(step_4_port_visit_events_df["port_visit"], sep="_"),
    ],
    axis=1,
)

In [42]:
step_4_port_visits_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 37 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   id                                             8 non-null      object 
 1   name                                           8 non-null      object 
 2   ssvid                                          8 non-null      object 
 3   flag                                           8 non-null      object 
 4   type                                           8 non-null      object 
 5   public_authorizations                          8 non-null      object 
 6   nextPort                                       0 non-null      object 
 7   visit_id                                       8 non-null      object 
 8   confidence                                     8 non-null      object 
 9   duration_hrs                                   8 non-null 

In [43]:
step_4_port_visits_df[
    [
        "name",
        "ssvid",
        "confidence",
        "start_anchorage_name",
        "intermediate_anchorage_name",
        "end_anchorage_name",
    ]
]

,name,ssvid,confidence,start_anchorage_name,intermediate_anchorage_name,end_anchorage_name
0,ATLANTIC SURF III,701024000,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
1,CAPESANTE,701006605,4,USHUAIA,USHUAIA,USHUAIA
2,CAPESANTE,701006605,4,USHUAIA,USHUAIA,USHUAIA
3,ATLANTIC SURF III,701024000,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
4,CAPESANTE,701006605,4,USHUAIA,USHUAIA,USHUAIA
5,ATLANTIC SURF III,701024000,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA
6,CAPESANTE,701006605,4,USHUAIA,USHUAIA,USHUAIA
7,ATLANTIC SURF III,701024000,4,MAR DEL PLATA,MAR DEL PLATA,MAR DEL PLATA


In [44]:
step_4_port_visits_df["ssvid"].value_counts()

ssvid
701024000    4
701006605    4
Name: count, dtype: int64

### What We have learned from  step 4

- **Apparent Fishing Events:**
  - `ATLANTIC SURF III (mmsi: 701024000, flag: ARG)`- has been detected in multiple apparent fishing events during the analyzed timeframe (August 2024 – January 2025)
  - `CAPESANTE (mmsi: 701006605, flag: ARG)` - has been detected in multiple apparent fishing events during the analyzed timeframe (August 2024 – January 2025)
- **Port Visit Events:**
  - `ATLANTIC SURF III (mmsi: 701024000, flag: ARG)`- potentially made multiple port visits, including stops at `MAR DEL PLATA`
  - `CAPESANTE (mmsi: 701006605, flag: ARG)` - potentially made multiple port visits, including stops at `USHUAIA`
- **ENCOUNTER Events:** No explicit **ENCOUNTER** events were returned in the response dataset. Check more details [here](https://globalfishingwatch.org/faqs/what-is-a-vessel-encounter/). You can read more about transshipment behavior from our [report](https://globalfishingwatch.org/wp-content/uploads/GlobalViewOfTransshipment_Aug2017.pdf) or [scientific publication](https://www.frontiersin.org/articles/10.3389/fmars.2018.00240/full).

**Potential Considerations:**

- The vessel's fishing activities appear near the EEZ boundary, requiring further assessment of compliance with national or RFMO regulations.
- The absence of matching public authorizations in the RFMO registry does not necessarily indicate illegality, but it suggests that authorities may need to verify through national databases or official sources.

### Summary of API Flow

1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** - Retrieve apparent fishing effort for **trawlers** within Argentinian EEZ.
2. **[Vessels API](https://globalfishingwatch.org/our-apis/documentation#vessels-api)** - Fetch **vessel identity**, **ownership history**, and **public authorizations**.
3. **[Events API](https://globalfishingwatch.org/our-apis/documentation#events-api)** - Detect potential **port visits**, **encounters**, and **apparent fishing** events to analyze operational patterns.
4. **Assess potential risks** - Compare registry records, AIS data, and inferred vessel activity for enforcement follow-ups.
5. **Generate a report** - Provide a structured analysis for relevant authorities.